# Studio 2 — Training Final di Google Colab GPU

**Kelompok:** BAI  
**Repo:** https://github.com/Aryasatyaa45/Studio2_BAI  
**Deadline Final:** 25 Oktober 2026, 23:59 WIB

## Panduan Singkat
1. Upload folder `train/` ke Google Drive (`MyDrive/Studio2/train/`)
2. Jalankan cell satu per satu dari atas ke bawah
3. Pastikan runtime GPU: **Runtime → Change runtime type → GPU**
4. File eksperimen sudah tersedia di repo, tidak perlu dibuat manual
5. Push ke GitHub di cell terakhir

## 1. Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repository kelompok
!git clone https://github.com/Aryasatyaa45/Studio2_BAI.git
%cd Studio2_BAI
!ls -la

## 2. Verifikasi GPU

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')
else:
    print('WARNING: GPU tidak terdeteksi! Ganti runtime ke GPU.')

## 3. Siapkan Dataset

In [ ]:
import shutil
from pathlib import Path

drive_data = Path('/content/drive/MyDrive/Studio2/train')
local_data = Path('/content/Studio2_BAI/dataset_train')

if not drive_data.exists():
    raise FileNotFoundError(f'Dataset tidak ditemukan di {drive_data}. Upload folder train/ ke Google Drive.')

if local_data.exists():
    shutil.rmtree(local_data)
shutil.copytree(drive_data, local_data)
print(f'Dataset disalin ke {local_data}')

for cls_dir in sorted(local_data.iterdir()):
    if cls_dir.is_dir():
        n = len(list(cls_dir.glob('*.jpg')))
        print(f'{cls_dir.name}: {n} gambar')

## 4. Install Dependencies

In [ ]:
!pip install -q torch torchvision torchinfo pillow numpy

---

## 5. EKSPERIMEN 1 — Baseline ResNet-18 (8 epoch)

In [ ]:
!python train.py --data /content/Studio2_BAI/dataset_train --epochs 8 --batch-size 64 --lr 3e-4

In [ ]:
import shutil
shutil.copy('submission/model.pth', 'submission/model_baseline.pth')
print('Baseline dibackup')

---

## 6. EKSPERIMEN 2 — ResNet-18 + Weighted Loss + Augmentasi Agresif (15 epoch)

In [ ]:
!python train_exp1_weighted.py --data /content/Studio2_BAI/dataset_train --epochs 15 --batch-size 64 --lr 3e-4 --patience 5

In [ ]:
import shutil
shutil.copy('submission/model.pth', 'submission/model_exp1.pth')
print('Exp1 dibackup')

---

## 7. EKSPERIMEN 3 — ResNet-34 Pretrained (12 epoch)

**PENTING:** Jika model ini terpilih, `submission/model.py` WAJIB diganti ke ResNet-34!

In [ ]:
!python train_exp2_resnet34.py --data /content/Studio2_BAI/dataset_train --epochs 12 --batch-size 64 --lr 3e-4 --patience 5

In [ ]:
import shutil
shutil.copy('submission/model.pth', 'submission/model_exp2.pth')
print('Exp2 dibackup')

---

## 8. Pilih Model Terbaik

In [ ]:
# Isi hasil Macro-F1 dari ketiga eksperimen di atas, lalu pilih yang terbaik
print('=== PERBANDINGAN EKSPERIMEN ===')
print('Eksperimen 1 (Baseline ResNet-18):    Macro-F1 = ???')
print('Eksperimen 2 (Weighted+Augmentasi):   Macro-F1 = ???')
print('Eksperimen 3 (ResNet-34):             Macro-F1 = ???')
print()
print('Pilih model terbaik, lalu uncomment baris yang sesuai di bawah:')

In [ ]:
import shutil

# Jika Eksperimen 1 terbaik (ResNet-18):
# shutil.copy('submission/model_baseline.pth', 'submission/model.pth')
# print('Model terbaik: Baseline ResNet-18')

# Jika Eksperimen 2 terbaik (ResNet-18 weighted):
# shutil.copy('submission/model_exp1.pth', 'submission/model.pth')
# print('Model terbaik: Exp1 ResNet-18 weighted')

# Jika Eksperimen 3 terbaik (ResNet-34):
# shutil.copy('submission/model_exp2.pth', 'submission/model.pth')
# shutil.copy('submission/model_resnet34.py', 'submission/model.py')
# print('Model terbaik: Exp2 ResNet-34 (model.py juga diganti)')

## 9. Evaluasi Final

In [ ]:
!python evaluate.py --submission ./submission --data ./local_data/my_val --out ./final_metrics.json

In [ ]:
import json
with open('final_metrics.json') as f:
    m = json.load(f)
print(f"Macro-F1: {m['macro_f1']:.4f}")
print(f"Accuracy: {m['accuracy']:.4f}")
for cls, f1 in m['per_class_f1'].items():
    print(f"  F1 {cls:<15} {f1:.4f}")

---

## 10. Push ke GitHub (FINAL SUBMISSION)

In [ ]:
# Konfigurasi Git
!git config --global user.email "aryasatya@student.itera.ac.id"
!git config --global user.name "Aryasatya"
!git status

In [ ]:
!git add submission/model.pth submission/model.py README.md
!git commit -m "Final submission: model terbaik Studio 2"
!git log --oneline -3

In [ ]:
# Push ke GitHub (gunakan Personal Access Token sebagai password)
!git push origin main

---

## Checklist Final

- [ ] `submission/model.py` ada dan sesuai arsitektur model terbaik
- [ ] `submission/model.pth` ada (ukuran ~40-90 MB)
- [ ] Evaluator lokal lolos dengan Macro-F1 terbaik
- [ ] Commit terakhir sebelum 25 Okt 2026, 23:59 WIB
- [ ] Push ke GitHub berhasil